In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.base import BaseEstimator, ClassifierMixin

# =======================
# Configurações principais
# =======================
DATA_DIR = Path('dados')
CSV_PATHS = {
    1: DATA_DIR / 'dados_modelo1.csv',
    2: DATA_DIR / 'dados_modelo2.csv',
    3: DATA_DIR / 'dados_modelo3.csv',
}
TARGET = 'Aprovou_Aprovou'
random_state_global = 42

# Regularização mais forte para generalizar melhor
candidate_depths = [3, 4, 6, 8]
n_estimators = 500
minority_label = 0.0
rf_params = dict(
    n_estimators=n_estimators,
    random_state=random_state_global,
    n_jobs=-1,
    class_weight='balanced',
    max_features='sqrt',         # corrigido: 'sqrt' em vez de 'auto'
    min_samples_leaf=5,
    min_samples_split=10
)

# Meta global de recall da minoria e preferência de threshold
RECALL_MIN_TARGET_GLOBAL = 0.50  # ajuste se necessário
THRESHOLD_PREFERENCE_DEFAULT = 'precision'  # 'precision' ou 'f1'

# Se quiser favorecer recall diretamente na seleção de threshold
USE_F_BETA_ON_FALLBACK = True
F_BETA = 2.0  # beta>1 favorece recall

# Métrica de tuning de profundidade: 'f1' (padrão) ou 'recall'
TUNING_METRIC = 'f1'

# Calibração opcional das probabilidades (custa tempo)
USE_CALIBRATION = False  # altere para True para calibrar
CALIBRATION_METHOD = 'isotonic'  # 'isotonic' (dados suficientes) ou 'sigmoid'
CALIBRATION_CV = 3

# =======================
# Funções auxiliares
# =======================
def build_preprocess(X):
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()
    return ColumnTransformer(
        transformers=[
            ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
            ('cat', Pipeline([
                ('imp', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore'))
            ]), cat_cols)
        ],
        remainder='drop'
    )

class ProbRF(BaseEstimator, ClassifierMixin):
    """
    Wrapper para RandomForest que permite calibração de probabilidade opcional.
    Mantém API predict_proba/predict.
    """
    def __init__(self, max_depth=None, use_calibration=False, calibration_method='isotonic', calibration_cv=3):
        self.max_depth = max_depth
        self.use_calibration = use_calibration
        self.calibration_method = calibration_method
        self.calibration_cv = calibration_cv
        self.rf_ = None
        self.clf_ = None
        self.classes_ = None

    def fit(self, X, y):
        rf = RandomForestClassifier(max_depth=self.max_depth, **rf_params)
        if self.use_calibration:
            clf = CalibratedClassifierCV(
                estimator=rf,              # parâmetro 'estimator' nas versões novas
                method=self.calibration_method,
                cv=self.calibration_cv
            )
            clf.fit(X, y)
            self.clf_ = clf
            self.classes_ = clf.classes_
        else:
            rf.fit(X, y)
            self.rf_ = rf
            self.classes_ = rf.classes_
        return self

    def predict_proba(self, X):
        if self.use_calibration:
            return self.clf_.predict_proba(X)
        return self.rf_.predict_proba(X)

    def predict(self, X):
        if self.use_calibration:
            return self.clf_.predict(X)
        return self.rf_.predict(X)

def make_pipe(X_subset, max_depth=None, use_calibration=False):
    preprocess = build_preprocess(X_subset)
    clf = ProbRF(
        max_depth=max_depth,
        use_calibration=use_calibration,
        calibration_method=CALIBRATION_METHOD,
        calibration_cv=CALIBRATION_CV
    )
    return Pipeline([('prep', preprocess), ('rf', clf)])

def evaluate_minority_metric(y_true, y_pred, metric='f1'):
    if metric == 'recall':
        return recall_score(y_true, y_pred, pos_label=minority_label, zero_division=0)
    return f1_score(y_true, y_pred, pos_label=minority_label, zero_division=0)

def fbeta_minority(y_true, y_pred, beta=1.0):
    prec = precision_score(y_true, y_pred, pos_label=minority_label, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=minority_label, zero_division=0)
    if prec == 0 and rec == 0:
        return 0.0
    beta2 = beta * beta
    return (1 + beta2) * (prec * rec) / (beta2 * prec + rec + 1e-12)

def tune_depth(X_train, y_train, X_val, y_val, depths, metric='f1', use_calibration=False):
    best, best_score, best_depth = None, -np.inf, None
    cols = X_train.columns  # usar todas as colunas
    for md in depths:
        pipe = make_pipe(X_train[cols], max_depth=md, use_calibration=use_calibration)
        pipe.fit(X_train[cols], y_train)
        y_val_pred = pipe.predict(X_val[cols])
        score = evaluate_minority_metric(y_val, y_val_pred, metric=metric)
        # desempate por menor profundidade (mais simples)
        if (score > best_score) or (np.isclose(score, best_score) and (best_depth is None or (md is not None and md < best_depth))):
            best, best_score, best_depth = pipe, score, md
    return best, best_depth, best_score

def predict_with_threshold(pipe, X, threshold):
    classes_ = pipe.named_steps['rf'].classes_
    idx_min = list(classes_).index(minority_label)
    proba_min = pipe.predict_proba(X)[:, idx_min]
    return np.where(proba_min >= threshold, minority_label, 1.0)

def choose_threshold_with_recall_target(pipe, X_val, y_val, recall_target, preference='precision'):
    classes_ = pipe.named_steps['rf'].classes_
    idx_min = list(classes_).index(minority_label)
    proba_min = pipe.predict_proba(X_val)[:, idx_min]

    thresholds = np.linspace(0, 1, 201)
    candidates = []
    for thr in thresholds:
        y_pred = np.where(proba_min >= thr, minority_label, 1.0)
        rec_min = recall_score(y_val, y_pred, pos_label=minority_label, zero_division=0)
        prec_min = precision_score(y_val, y_pred, pos_label=minority_label, zero_division=0)
        f1_min = f1_score(y_val, y_pred, pos_label=minority_label, zero_division=0)
        fbeta_min = fbeta_minority(y_val, y_pred, beta=F_BETA) if USE_F_BETA_ON_FALLBACK else f1_min
        candidates.append((thr, rec_min, prec_min, f1_min, fbeta_min))

    feasible = [c for c in candidates if c[1] >= recall_target]
    if not feasible:
        # Ninguém bate a meta → priorize maior recall e, como desempate, melhor F_beta
        feasible = sorted(candidates, key=lambda x: (x[1], x[4]), reverse=True)[:5]

    if preference == 'precision':
        best = max(feasible, key=lambda x: (x[2], x[1], x[3]))  # precisão, depois recall, depois F1
    else:
        best = max(feasible, key=lambda x: (x[3], x[1], x[2]))  # F1, depois recall, depois precisão

    thr, rec_min, prec_min, f1_min, fbeta_min = best
    return thr, {'rec_min': rec_min, 'prec_min': prec_min, 'f1_min': f1_min, 'fbeta_min': fbeta_min}

def compute_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_minority': recall_score(y_true, y_pred, pos_label=minority_label, zero_division=0),
        'precision_minority': precision_score(y_true, y_pred, pos_label=minority_label, zero_division=0),
        'f1_minority': f1_score(y_true, y_pred, pos_label=minority_label, zero_division=0),
        'fbeta_minority': fbeta_minority(y_true, y_pred, beta=F_BETA),
    }

# =======================
# Execução por tabela
# =======================
all_summaries = []

for table_id, csv_path in CSV_PATHS.items():
    print(f"\n======================")
    print(f"Tabela {table_id}: {csv_path}")
    print(f"======================")

    if not Path(csv_path).exists():
        print(f"Arquivo não encontrado: {csv_path}. Pulando.")
        continue

    df = pd.read_csv(csv_path)

    if TARGET not in df.columns:
        print(f"Coluna alvo '{TARGET}' não encontrada em {csv_path}. Pulando.")
        continue

    y = df[TARGET]
    X = df.drop(columns=[TARGET])

    # Splits estratificados
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=random_state_global, stratify=y
    )
    X_test, X_val, y_test, y_val = train_test_split(
        X_temp, y_temp, test_size=1/3, random_state=random_state_global, stratify=y_temp
    )
    print(f"Formas: treino={X_train.shape}, teste={X_test.shape}, val={X_val.shape}")

    # Tuning de profundidade usando todas as colunas (com calibração opcional)
    pipe, best_depth, val_min_metric = tune_depth(
        X_train, y_train, X_val, y_val, candidate_depths, metric=TUNING_METRIC, use_calibration=USE_CALIBRATION
    )

    # Preferência de threshold adaptativa
    threshold_preference = THRESHOLD_PREFERENCE_DEFAULT

    thr_tmp, val_stats_tmp = choose_threshold_with_recall_target(
        pipe, X_val[X_val.columns], y_val, RECALL_MIN_TARGET_GLOBAL, preference=threshold_preference
    )
    if np.isclose(val_stats_tmp['prec_min'], 1.0):
        threshold_preference = 'f1'

    thr, val_stats = choose_threshold_with_recall_target(
        pipe, X_val[X_val.columns], y_val, RECALL_MIN_TARGET_GLOBAL, preference=threshold_preference
    )

    # Avaliação no teste
    y_pred_test = predict_with_threshold(pipe, X_test[X_test.columns], thr)
    metrics = compute_metrics(y_test, y_pred_test)

    print(f"\n[Tabela {table_id}] max_depth={best_depth} | thr={thr:.3f} | "
          f"meta_recall_global={RECALL_MIN_TARGET_GLOBAL:.2f} | tuning_metric={TUNING_METRIC} | "
          f"pref_threshold={threshold_preference} | calibrated={USE_CALIBRATION}")
    print(f"Val (threshold): recall_min={val_stats['rec_min']:.3f}, "
          f"precision_min={val_stats['prec_min']:.3f}, f1_min={val_stats['f1_min']:.3f}, "
          f"f{F_BETA:.0f}_min={val_stats['fbeta_min']:.3f}")
    print(f"Teste: Acc={metrics['accuracy']:.4f}, Recall_min={metrics['recall_minority']:.4f}, "
          f"Prec_min={metrics['precision_minority']:.4f}, F1_min={metrics['f1_minority']:.4f}, "
          f"F{F_BETA:.0f}_min={metrics['fbeta_minority']:.4f}")
    print("Relatório por classe (teste):")
    print(classification_report(y_test, y_pred_test, digits=4, zero_division=0))
    print("Matriz de confusão (teste):")
    print(confusion_matrix(y_test, y_pred_test))

    all_summaries.append({
        'table_id': table_id,
        'best_depth': best_depth,
        'thr': thr,
        'pref_threshold': threshold_preference,
        'calibrated': USE_CALIBRATION,
        'val_min_metric_tuned': val_min_metric,
        'val_recall_min_thr': val_stats['rec_min'],
        'val_precision_min_thr': val_stats['prec_min'],
        'val_f1_min_thr': val_stats['f1_min'],
        f'val_f{int(F_BETA)}_min_thr': val_stats['fbeta_min'],
        **metrics
    })

# Resumo geral
if all_summaries:
    df_all = pd.DataFrame(all_summaries).sort_values(['table_id'])
    print("\n======================")
    print("Resumo geral (por tabela):")
    cols_to_show = [
        'table_id','best_depth','thr','pref_threshold','calibrated',
        'val_min_metric_tuned','val_recall_min_thr','val_precision_min_thr','val_f1_min_thr', f'val_f{int(F_BETA)}_min_thr',
        'accuracy','recall_macro','recall_minority','precision_minority','f1_minority','fbeta_minority'
    ]
    print(df_all[cols_to_show])
else:
    print("\nNenhum resumo gerado. Verifique os arquivos e as colunas alvo/feature.")


Tabela 1: dados\dados_modelo1.csv
Formas: treino=(962, 9), teste=(275, 9), val=(138, 9)

[Tabela 1] max_depth=6 | thr=0.620 | meta_recall_global=0.50 | tuning_metric=f1 | pref_threshold=precision | calibrated=False
Val (threshold): recall_min=0.500, precision_min=0.500, f1_min=0.500, f2_min=0.500
Teste: Acc=0.9127, Recall_min=0.3125, Prec_min=0.2778, F1_min=0.2941, F2_min=0.3049
Relatório por classe (teste):
              precision    recall  f1-score   support

         0.0     0.2778    0.3125    0.2941        16
         1.0     0.9572    0.9498    0.9535       259

    accuracy                         0.9127       275
   macro avg     0.6175    0.6312    0.6238       275
weighted avg     0.9177    0.9127    0.9151       275

Matriz de confusão (teste):
[[  5  11]
 [ 13 246]]

Tabela 2: dados\dados_modelo2.csv
Formas: treino=(962, 12), teste=(275, 12), val=(138, 12)

[Tabela 2] max_depth=4 | thr=0.495 | meta_recall_global=0.50 | tuning_metric=f1 | pref_threshold=precision | calibra